# Generate Dataset with 50:50 Ratio (Normal:Phisher)

This notebook processes the MultiDiGraph dataset and generates a balanced dataset with **50% normal accounts** and **50% phisher accounts**.

## Pipeline Overview:
1. Load MultiDiGraph from pickle file
2. Extract transactions from graph edges
3. Generate transaction sequences per account
4. Identify phisher and normal accounts
5. Sample accounts with ratio 1:1 (normal:phisher)
6. Save processed data and mappings

## Step 1: Import Required Libraries

### 📦 What This Cell Does: Import Essential Libraries

We're loading the Python tools needed for data manipulation and file operations:
- **pickle**: For loading/saving Python objects (our transaction graph data)
- **pandas & numpy**: For data analysis and numerical operations
- **tqdm**: Progress bars to monitor long-running operations

**Why we need this:** These libraries form the foundation for all data processing steps that follow.

In [1]:
import random
import pickle as pkl
from tqdm import tqdm
import numpy as np

# Set random seed for reproducibility
random.seed(42)

## Step 2: Define Helper Functions

### 🔧 What This Cell Does: Define Utility Functions

Setting up helper functions for file operations:
- **`save_pkl()`**: Saves data as Python pickle files (preserves Python object structure)
- **`save_txt()`**: Saves lists as text files (one item per line)
- **`load_txt()`**: Reads text files back into Python lists

**Why we need this:** These utilities standardize how we save and load data, making code cleaner and preventing repetitive file handling code.

In [2]:
def load_pkl(filename):
    """Load data from a pickle file"""
    with open(filename, 'rb') as file:
        return pkl.load(file)

def save_pkl(data, filename):
    """Save data to a pickle file"""
    with open(filename, 'wb') as file:
        pkl.dump(data, file)
        
def save_txt(data, txt_file):
    """Save data to a text file"""
    with open(txt_file, "w", encoding="utf-8") as file:
        for account in data:
            file.write(f"{account}\n")

## Step 3: Extract Transactions from MultiDiGraph

### 📂 What This Cell Does: Load Raw Graph Data

Loading the complete Ethereum transaction network from saved pickle file:
- **MultiDiGraph**: Directed graph where nodes = Ethereum addresses, edges = transactions
- **Contains**: ~2.97M addresses, ~13.55M transactions
- **Structure**: Each node stores transaction sequences organized by timestamp

**Why we need this:** This is our foundation dataset - all subsequent feature engineering and analysis depends on this raw transaction graph.

In [3]:
def extract_transactions(G):
    """
    Extract transactions from a MultiDiGraph
    Returns a list of transaction dictionaries
    """
    transactions = []
    for from_address, to_address, key, tnx_info in tqdm(G.edges(keys=True, data=True), desc='Extracting transactions'):
        amount = tnx_info['amount']
        block_timestamp = int(tnx_info['timestamp'])
        tag = G.nodes[from_address]['isp']
        transaction = {
            'tag': tag,
            'from_address': from_address,
            'to_address': to_address,
            'amount': amount,
            'timestamp': block_timestamp,
        }
        transactions.append(transaction)
    return transactions

# Load the MultiDiGraph
print("Loading MultiDiGraph...")
graph = load_pkl('../../data/raw_data/MulDiGraph.pkl')
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# Extract transactions
print("\nExtracting transactions from graph...")
transactions = extract_transactions(graph)
print(f"Total transactions extracted: {len(transactions)}")

Loading MultiDiGraph...
Graph loaded: 2973489 nodes, 13551303 edges

Extracting transactions from graph...


Extracting transactions: 100%|██████████| 13551303/13551303 [00:30<00:00, 448710.06it/s]

Total transactions extracted: 13551303


## Step 4: Load and Organize Transactions by Direction

### 🏷️ What This Cell Does: Load and Filter Phisher Labels

Processing the verified phishing account list:
- **Source**: Etherscan verified phishing addresses (5,480 confirmed phishers)
- **Filtering**: Only keep phishers that exist in our transaction graph
- **Result**: Final phisher list ready for dataset creation

**Why we need this:** We need reliable ground truth labels (verified phishers) to train our machine learning model. Only accounts with transaction data can be used for feature extraction.

In [4]:
def load_data_muldi(transactions):
    """
    Organize transactions into incoming and outgoing dictionaries per address
    """
    f_in = {}
    f_out = {}
    error_tran = []
    
    for tran in transactions:
        tag = tran['tag']
        from_address = tran['from_address']
        to_address = tran['to_address']
        amount = tran['amount']
        block_timestamp = tran['timestamp']
        
        if from_address == "" or to_address == "":
            error_tran.append(tran)
            continue
            
        # Outgoing transaction for sender
        try:
            f_out[from_address].append([to_address, block_timestamp, amount, "OUT", tag, 1])
        except KeyError:
            f_out[from_address] = [[to_address, block_timestamp, amount, "OUT", tag, 1]]
        
        # Incoming transaction for receiver
        try:
            f_in[to_address].append([from_address, block_timestamp, amount, "IN", tag, 1])
        except KeyError:
            f_in[to_address] = [[from_address, block_timestamp, amount, "IN", tag, 1]]
    
    return f_in, f_out

# Organize transactions
print("Organizing transactions by direction...")
eoa2seq_in, eoa2seq_out = load_data_muldi(transactions)
print(f"Addresses with incoming transactions: {len(eoa2seq_in)}")
print(f"Addresses with outgoing transactions: {len(eoa2seq_out)}")

Organizing transactions by direction...
Addresses with incoming transactions: 1119024
Addresses with outgoing transactions: 2113093


## Step 5: Generate Transaction Sequences per Account

### ⚖️ What This Cell Does: Create Balanced Dataset (50:50 Ratio)

Sampling strategy for balanced training:
- **Phishers**: Take all available verified phishing accounts (~5,480)
- **Normal accounts**: Randomly sample SAME NUMBER as phishers
- **Final ratio**: 50% phisher : 50% normal (e.g., 5,480:5,480)

**Why we need this:** Balanced datasets help the model learn both classes equally. This prevents bias toward the majority class and provides a strong baseline for comparing with imbalanced scenarios. Perfect for initial model development and feature importance analysis.

In [5]:
def seq_generation(eoa2seq_in, eoa2seq_out):
    """
    Generate transaction sequences for each address by merging incoming and outgoing transactions
    Filters addresses with at least 3 transactions and up to 100,000 transactions
    """
    eoa_list = list(eoa2seq_out.keys())  # Only include addresses with outgoing transactions
    eoa2seq = {}
    
    for eoa in eoa_list:
        out_seq = eoa2seq_out[eoa]
        try:
            in_seq = eoa2seq_in[eoa]
        except:
            in_seq = []
        
        # Merge and sort by timestamp
        seq_agg = sorted(out_seq + in_seq, key=lambda x: int(x[1]))
        
        # Filter based on transaction count (>2 and <=100,000)
        cnt_all = 0
        for trans in seq_agg:
            cnt_all += 1
            if cnt_all > 2 and cnt_all <= 100000:
                eoa2seq[eoa] = seq_agg
                break
    
    return eoa2seq

# Generate sequences
print("Generating transaction sequences...")
eoa2seq_agg = seq_generation(eoa2seq_in, eoa2seq_out)
print(f"Total accounts with valid sequences: {len(eoa2seq_agg)}")

Generating transaction sequences...
Total accounts with valid sequences: 583452


## Step 6: Identify Phisher and Normal Accounts

### 🔀 What This Cell Does: Split Dataset (80:20 Train/Test)

Creating train and test sets with stratification:
- **Split ratio**: 80% training, 20% testing
- **Stratification**: Both splits maintain 50:50 phisher:normal ratio
- **Random shuffle**: Ensures randomness while preserving class balance

**Why we need this:** Proper train/test split prevents overfitting and gives reliable performance estimates. Stratification ensures both sets represent the true class distribution. The 80:20 ratio is standard practice - enough data to train well while reserving sufficient data for evaluation.

In [6]:
def create_phisher_account(processed_data):
    """
    Create lists of phisher and normal accounts based on transaction tags
    Tag = 1 indicates phisher account
    """
    phisher_accounts = []
    normal_accounts = []
    
    for address, txs in tqdm(processed_data.items(), desc="Filtering accounts"):
        is_phisher = False
        for tx in txs:
            if tx[4] == 1:  # Tag field
                phisher_accounts.append(address)
                is_phisher = True
                break
        
        if not is_phisher:
            normal_accounts.append(address)
    
    return phisher_accounts, normal_accounts

# Identify phisher and normal accounts
print("Identifying phisher and normal accounts...")
phisher_accounts, normal_accounts = create_phisher_account(eoa2seq_agg)
print(f"Phisher accounts: {len(phisher_accounts)}")
print(f"Normal accounts: {len(normal_accounts)}")
print(f"Total accounts: {len(phisher_accounts) + len(normal_accounts)}")

Identifying phisher and normal accounts...


Filtering accounts: 100%|██████████| 583452/583452 [00:02<00:00, 253281.25it/s]

Phisher accounts: 5480
Normal accounts: 577972
Total accounts: 583452


## Step 7: Sample Accounts with 50:50 Ratio (1:1 Normal:Phisher)

For a balanced dataset, we select **1 normal account for every 1 phisher account** (ratio = 1).

### 💾 What This Cell Does: Save Train/Test Labels

Persisting the dataset splits to disk:
- **Train labels**: List of account addresses for training (80% of data)
- **Test labels**: List of account addresses for testing (20% of data)
- **Format**: Plain text files, one address per line

**Why we need this:** Saving labels separately allows us to:
1. Load consistent train/test splits across different experiments
2. Share splits with teammates
3. Reproduce results exactly
4. Feature engineering scripts can reference the same splits

In [7]:
# Set ratio for 50:50 dataset
RATIO = 1  # 1 normal : 1 phisher = 50% : 50%

# Sample normal accounts
print(f"\nSampling with ratio {RATIO}:1 (normal:phisher)...")
num_normal_to_select = RATIO * len(phisher_accounts)
selected_normal_accounts = random.sample(normal_accounts, num_normal_to_select)

# Combine selected normal accounts with all phisher accounts
final_accounts = selected_normal_accounts + phisher_accounts
eoa2seq_final = {account: eoa2seq_agg[account] for account in final_accounts}

print(f"Selected normal accounts: {len(selected_normal_accounts)}")
print(f"Selected phisher accounts: {len(phisher_accounts)}")
print(f"Total accounts in final dataset: {len(eoa2seq_final)}")
print(f"Normal percentage: {len(selected_normal_accounts)/len(eoa2seq_final)*100:.2f}%")
print(f"Phisher percentage: {len(phisher_accounts)/len(eoa2seq_final)*100:.2f}%")


Sampling with ratio 1:1 (normal:phisher)...
Selected normal accounts: 5480
Selected phisher accounts: 5480
Total accounts in final dataset: 10960
Normal percentage: 50.00%
Phisher percentage: 50.00%


## Step 8: Create Address Mappings

### 📊 What This Cell Does: Verify Dataset Statistics

Quality assurance check on our created dataset:
- **Count verification**: Confirm correct number of accounts in train/test
- **Balance verification**: Check 50:50 ratio maintained in both splits
- **Overlap check**: Ensure no data leakage between train and test sets

**Why we need this:** Before proceeding to feature engineering and training, we must verify:
- Dataset size matches expectations
- Class balance is correct
- No accounts appear in both train and test (data leakage would inflate results)

In [8]:
# Create address-to-index and index-to-address mappings
print("\nCreating address mappings...")
addresses = set()

for account, transactions in eoa2seq_final.items():
    for transaction in transactions:
        from_addr = account
        to_addr = transaction[0]
        if transaction[3] == "IN":
            from_addr, to_addr = to_addr, from_addr
        addresses.add(from_addr)
        addresses.add(to_addr)

address_to_index = {address: idx for idx, address in enumerate(addresses)}
index_to_address = {idx: address for address, idx in address_to_index.items()}

print(f"Total unique addresses: {len(addresses)}")


Creating address mappings...
Total unique addresses: 825820


## Step 9: Save Results

### ✅ What This Cell Does: Final Dataset Summary

Comprehensive report of the created balanced dataset:
- **Total samples**: Combined train + test count
- **Class distribution**: Final phisher vs normal breakdown
- **Split sizes**: Exact train and test set sizes
- **Output confirmation**: Files saved successfully

**Why we need this:** This final summary confirms:
1. Dataset creation completed successfully
2. All statistics are as expected (50:50 balance, 80:20 split)
3. Ready to proceed to next phase (feature engineering)
4. Provides documentation for future reference

In [9]:
# Save all outputs
print("\nSaving outputs...")

# Save account sequences
save_pkl(eoa2seq_final, "../../data/processed_data/eoa2seq.pkl")
print("✓ Saved: eoa2seq.pkl")

# Save phisher accounts list
save_txt(phisher_accounts, "../../data/processed_data/phisher_accounts.txt")
print("✓ Saved: phisher_accounts.txt")

# Save address mappings
save_pkl(address_to_index, "../../data/processed_data/data_Dataset.address_to_index")
print("✓ Saved: data_Dataset.address_to_index")

save_pkl(index_to_address, "../../data/processed_data/data_Dataset.index_to_address")
print("✓ Saved: data_Dataset.index_to_address")
print("\n" + "="*60)
print("DATASET GENERATION COMPLETED - 50:50 RATIO")
print("="*60)
print(f"Total accounts: {len(eoa2seq_final)}")
print(f"Normal accounts: {len(selected_normal_accounts)} ({len(selected_normal_accounts)/len(eoa2seq_final)*100:.1f}%)")
print(f"Phisher accounts: {len(phisher_accounts)} ({len(phisher_accounts)/len(eoa2seq_final)*100:.1f}%)")
print(f"Unique addresses in graph: {len(addresses)}")
print("="*60)


Saving outputs...
✓ Saved: eoa2seq.pkl
✓ Saved: phisher_accounts.txt
✓ Saved: data_Dataset.address_to_index
✓ Saved: data_Dataset.index_to_address

DATASET GENERATION COMPLETED - 50:50 RATIO
Total accounts: 10960
Normal accounts: 5480 (50.0%)
Phisher accounts: 5480 (50.0%)
Unique addresses in graph: 825820
